**Name:** \_\_\_\_\_

**EID:** \_\_\_\_\_

# 教程 3：图像中的人脸检测

在本教程中，你将训练支持向量机（SVM）和核支持向量机（kernel SVM）分类器，用于检测一个小图像块（image patch）中是否包含人脸。

首先我们需要初始化 Python。运行下面的单元格。

**注意**：本教程推荐使用的 scikit-learn 版本为 **1.4.2**。如果使用其他版本，可能会出现 bug。

In [ ]:
%matplotlib inline
import IPython.core.display
# setup output image format (Chrome works best)
IPython.core.display.set_matplotlib_formats("svg")
import matplotlib.pyplot as plt
import matplotlib
from numpy import *
from sklearn import *
import os
import zipfile
import fnmatch
random.seed(4487)
from scipy import ndimage
from scipy import signal
import skimage.color
import skimage.exposure
import skimage.io
import skimage.util
import skimage

## 1. 加载数据与预处理
接下来我们需要加载图像。下载 `faces.zip`，并把它放在与本 ipynb 文件相同的目录下。**不要解压该文件。**然后运行下面的单元格来加载图像。

In [ ]:
imgdata = {'train':[], 'test':[]}
classes = {'train':[], 'test':[]}

# the dataset is too big, so subsample the training and test sets...
# reduce training set by a factor of 4
train_subsample = 4
train_counter = [0, 0]
# maximum number of samples in each class for test set
test_maxsample = 472
test_counter = [0, 0]

# load the zip file
filename = 'faces.zip'
zfile = zipfile.ZipFile(filename, 'r')

for name in zfile.namelist():
    # check file name matches
    if fnmatch.fnmatch(name, "faces/*/*/*.png"):

        # filename is : faces/train/face/fname.png
        (fdir1, fname)  = os.path.split(name)     # get file name
        (fdir2, fclass) = os.path.split(fdir1) # get class (face, nonface)
        (fdir3, fset)   = os.path.split(fdir2) # get training/test set
        # class 1 = face; class 0 = non-face
        myclass = int(fclass == "face")

        loadme = False
        if fset == 'train':
            if (train_counter[myclass] % train_subsample) == 0:
                loadme = True
            train_counter[myclass] += 1
        elif fset == 'test':
            if test_counter[myclass] < test_maxsample:
                loadme = True
            test_counter[myclass] += 1

        if (loadme):
            # open file in memory, and parse as an image
            myfile = zfile.open(name)
            #img = matplotlib.image.imread(myfile)
            img = skimage.io.imread(myfile)
            # convert to grayscale
            img = skimage.color.rgb2gray(skimage.color.rgba2rgb(img))
            myfile.close()

            # append data
            imgdata[fset].append(img)
            classes[fset].append(myclass)


zfile.close()
imgsize = img.shape

print(len(imgdata['train']))
print(len(imgdata['test']))
trainclass2start = sum(classes['train'])

每张图像都是一个 $19\times19$ 的像素值数组。运行下面的代码查看一个示例：

In [ ]:
print(img.shape)
plt.subplot(1,2,1)
plt.imshow(imgdata['train'][0], cmap='gray', interpolation='nearest')
plt.title("face sample")
plt.subplot(1,2,2)
plt.imshow(imgdata['train'][trainclass2start], cmap='gray', interpolation='nearest')
plt.title("non-face sample")
plt.show()

运行下面的代码查看更多图像！

In [ ]:
# function to make an image montage
def image_montage(X, imsize=None, maxw=10):
    """X can be a list of images, or a matrix of vectorized images.
      Specify imsize when X is a matrix."""
    tmp = []
    numimgs = len(X)

    # create a list of images (reshape if necessary)
    for i in range(0,numimgs):
        if imsize != None:
            tmp.append(X[i].reshape(imsize))
        else:
            tmp.append(X[i])

    # add blanks
    if (numimgs > maxw) and (mod(numimgs, maxw) > 0):
        leftover = maxw - mod(numimgs, maxw)
        meanimg = 0.5*(X[0].max()+X[0].min())
        for i in range(0,leftover):
            tmp.append(ones(tmp[0].shape)*meanimg)

    # make the montage
    tmp2 = []
    for i in range(0,len(tmp),maxw):
        tmp2.append( hstack(tmp[i:i+maxw]) )
    montimg = vstack(tmp2)
    return montimg

# show a few images
plt.figure(figsize=(9,9))
plt.imshow(image_montage(imgdata['train'][::20]), cmap='gray', interpolation='nearest')
plt.show()

每张图像是一个二维数组，但分类器算法处理的是一维向量。运行下面的代码，通过"展平"（flattening）把所有图像转换成一维向量。结果应该是一个矩阵，其中每一行是一张展平后的图像。

In [ ]:
trainX = empty((len(imgdata['train']), prod(imgsize)))
for i,img in enumerate(imgdata['train']):
    trainX[i,:] = ravel(img)
trainY = asarray(classes['train'])  # convert list to numpy array
print(trainX.shape)
print(trainY.shape)

testX = empty((len(imgdata['test']), prod(imgsize)))
for i,img in enumerate(imgdata['test']):
    testX[i,:] = ravel(img)
testY = asarray(classes['test'])  # convert list to numpy array
print(testX.shape)
print(testY.shape)

## 2. 使用像素值进行检测

训练使用 RBF 核或多项式（polynomial）核的核 SVM 分类器，把一个图像块分类为人脸（face）或非人脸（non-face）。在测试集上评估所有分类器。

对特征做归一化，并设置好所有的参数和模型。

In [ ]:
# Normalization: sklearn.preprocessing.MinMaxScaler()
scaler = preprocessing.MinMaxScaler(feature_range=(-1,1))    # make scaling object
trainXn = scaler.fit_transform(trainX)   # use training data to fit scaling parameters
testXn  = scaler.transform(testX)        # apply scaling to test data

clfs = {}

In [ ]:
# setup all the parameters and models
exps = {
    'svm-lin': {
        'paramgrid': {'C': logspace(-2,3,10)},
        'clf': svm.SVC(kernel='linear') },
    'svm-rbf': {
        'paramgrid': {'C': logspace(-2,3,10), 'gamma': logspace(-4,3,10) },
        'clf': svm.SVC(kernel='rbf') },
    'svm-poly': {
        'paramgrid': {'C': logspace(-2,3,10), 'degree': [2, 3, 4] },
        'clf': svm.SVC(kernel='poly') },
    }

尝试训练每个分类器，并显示其（最优）参数。

In [ ]:
### 在此处填写你的代码
## 提示
## 1. 分类器：sklearn.model_selection.GridSearchCV()

计算每个分类器的训练准确率和测试准确率。

In [ ]:
### 在此处填写你的代码
## 提示
# 1. clf.predict()
# 2. metrics.accuracy_score()

In [ ]:
# set variables for later
predY = predYtest['svm-poly']
#adaclf = clfs['ada'].best_estimator_
svmclf_rbf = clfs['svm-rbf'].best_estimator_
svmclf_poly = clfs['svm-poly'].best_estimator_
#rfclf  = clfs['rf'].best_estimator_

_哪个分类器最好？_
- **在此填写你的答案**

### 错误分析
准确率只能反映分类器性能的一部分。我们还可以进一步考察分类器所犯错误的不同类型：
- _真正例 (True Positive, TP)_：分类器正确地判断为人脸
- _真负例 (True Negative, TN)_：分类器正确地判断为非人脸
- _假正例 (False Positive, FP)_：分类器判断为人脸，但实际不是人脸
- _假负例 (False Negative, FN)_：分类器判断为非人脸，但实际是人脸

这些可以用下表来概括：

<table>
<tr><th colspan=2 rowspan=2><th colspan=2 style="text-align: center">实际类别</th></tr>
<tr>  <th>人脸</th><th>非人脸</th></tr>
<tr><th rowspan=2>预测结果</th><th>人脸</th><td>真正例 (TP)</td><td>假正例 (FP)</td></tr>
<tr>  <th>非人脸</th><td>假负例 (FN)</td><td>真负例 (TN)</td></tr>
</table>

然后我们可以考察 _真正例率_ 和 _假正例率_：
- _真正例率 (TPR)_：真实人脸中被正确检测出来的比例
- _假正例率 (FPR)_：非人脸中被错误分类为人脸的比例。

使用下面的代码计算你的分类器的 TPR 和 FPR。

In [ ]:
# predY is the prediction from the classifier

Pind = where(testY==1) # indicies for face
Nind = where(testY==0) # indicies for non-face

TP = count_nonzero(testY[Pind] == predY[Pind])
FN = count_nonzero(testY[Pind] != predY[Pind])
TN = count_nonzero(testY[Nind] == predY[Nind])
FP = count_nonzero(testY[Nind] != predY[Nind])

TPR = TP / (TP+FN)
FPR = FP / (FP+TN)

print("TP=", TP)
print("FP=", FP)
print("TN=", TN)
print("FN=", FN)
print("TPR=", TPR)
print("FPR=", FPR)

_分类器是如何犯错的？_
- **在此填写你的答案**

对于核 SVM，我们可以查看支持向量（support vectors），看看分类器觉得哪些样本"难分类"。

In [ ]:
# svmclf is the trained SVM classifier

print("num support vectors:", len(svmclf_poly.support_vectors_))
si  = svmclf_poly.support_  # get indicies of support vectors

# get all the patches for each support vector
simg = [ imgdata['train'][i] for i in si ]

# make montage
outimg = image_montage(simg, maxw=20)

plt.figure(figsize=(9,9))
plt.imshow(outimg, cmap='gray', interpolation='nearest')

请对你观察到的、SVM 觉得困难的样本（即位于决策边界上或间隔内的样本）进行评论。
- **在此填写你的答案**

## 3. 使用图像特征进行检测

### 图像特征提取
使用像素值时，检测性能并不太好。问题在于我们把原始像素值当作特征，这使得分类器很难解读人脸中那些可能很重要的较大结构。为了解决这个问题，我们将使用一组滤波器从图像中提取特征。

运行下面的代码查看滤波器的输出。这些滤波器是一组黑色和白色的方块，它们会对图像中相似的结构产生响应。将滤波器应用到图像后，滤波器响应图会在 $4\times4$ 的窗口上进行聚合。因此每个滤波器产生 $5\times5$ 的特征响应。由于共有 $4$ 个滤波器，所以特征向量是 $100$ 维的。

In [ ]:
def extract_features(imgs, doplot=False):
    # the filter layout
    lay = [array([-1,1]), array([-1,1,-1]),
               array([[1],[-1]]), array([[-1],[1],[-1]])]
    sc=8            # size of each filter patch
    poolmode = 'i'  # pooling mode (interpolate)
    cmode = 'same'  # convolution mode
    brick = ones((sc,sc))  # filter patch
    ks = []
    for l in lay:
        tmp = [brick*i for i in l]
        if (l.ndim==1):
            k = hstack(tmp)
        else:
            k = vstack(tmp)
        ks.append(k)

    # get the filter response size
    if (poolmode=='max') or (poolmode=='absmax'):
        tmpimg = maxpool(maxpool(imgs[0]))
    else:
        tmpimg = ndimage.interpolation.zoom(imgs[0], 0.25)
    fs = prod(tmpimg.shape)

    # get the total feature length
    fst = fs*len(ks)

    # filter the images
    X  = empty((len(imgs), fst))
    for i,img in enumerate(imgs):
        x = empty(fst)

        # for each filter
        for j,th in enumerate(ks):
            # filter the image
            imgk = signal.convolve(img, ks[j], mode=cmode)

            # do pooling
            if poolmode == 'maxabs':
                mimg = maxpool(maxpool(abs(imgk)))
            elif poolmode == 'max':
                mimg = maxpool(maxpool(imgk))
            else:
                mimg = ndimage.interpolation.zoom(imgk, 0.25)

            # put responses into feature vector
            x[(j*fs):(j+1)*fs] = ravel(mimg)

            if (doplot):
                plt.subplot(3,len(ks),j+1)
                plt.imshow(ks[j], cmap='gray', interpolation='nearest')
                plt.title("filter " + str(j))
                plt.subplot(3,len(ks),len(ks)+j+1)
                plt.imshow(imgk, cmap='gray', interpolation='nearest')
                plt.title("filtered image")
                plt.subplot(3,len(ks),2*len(ks)+j+1)
                plt.imshow(mimg, cmap='gray', interpolation='nearest')
                plt.title("image features")
        X[i,:] = x

    return X

In [ ]:
# new features
img = imgdata['train'][0]
plt.imshow(img, cmap='gray', interpolation='nearest')
plt.title("image")
plt.figure(figsize=(9,9))
extract_features([img], doplot=True);

现在让我们在训练集和测试集上提取图像特征。这可能需要几秒钟。

In [ ]:
trainXf = extract_features(imgdata['train'])
print(trainXf.shape)
testXf = extract_features(imgdata['test'])
print(testXf.shape)

### 检测
现在在图像特征数据上训练 AdaBoost 和 SVM 分类器。在测试集上进行评估。

In [ ]:
### Nomalization
scalerf = preprocessing.MinMaxScaler(feature_range=(-1,1))    # make scaling object
trainXfn = scalerf.fit_transform(trainXf)   # use training data to fit scaling parameters
testXfn  = scalerf.transform(testXf)        # apply scaling to test data

clfs2 = {}

In [ ]:
# setup all the parameters and models
exps = {
    'svm-lin': {
        'paramgrid': {'C': logspace(-2,3,10)},
        'clf': svm.SVC(kernel='linear') },
    'svm-rbf': {
        'paramgrid': {'C': logspace(-2,3,10), 'gamma': logspace(-4,3,10) },
        'clf': svm.SVC(kernel='rbf') },
    'svm-poly': {
        'paramgrid': {'C': logspace(-2,3,10), 'degree': [2, 3, 4] },
        'clf': svm.SVC(kernel='poly') },
    }
clfs2 = {}

尝试训练每个分类器，并显示其（最优）参数。

In [ ]:
### 在此处填写你的代码
## 提示
## 分类器：sklearn.model_selection.GridSearchCV()

计算每个分类器的训练准确率和测试准确率。

In [ ]:
### 在此处填写你的代码

In [ ]:
testY = predYtest['svm-rbf']
bestclf = clfs2['svm-rbf']

### 错误分析
与前面类似，对新训练的分类器重复进行错误分析。

In [ ]:
### 在此处填写你的代码

使用图像特征的分类器有什么改进？
- **在此填写你的答案**

# 测试图像
现在让我们在一张真实图像上尝试你的人脸检测器。下载 "nasa-small.png" 图像，并把它放在与你的 ipynb 文件相同的目录下。下面的代码会加载图像、裁剪出图像块（patches），然后提取特征。（这可能需要几分钟）

In [ ]:
fname = "nasa-small.png"

In [ ]:
# load image
testimg3 = skimage.io.imread(fname)

# convert to grayscale
testimg = skimage.color.rgb2gray(testimg3)
print(testimg.shape)
plt.imshow(testimg, cmap='gray')

In [ ]:
# step size for the sliding window
step = 4

# extract window patches with step size of 4
patches = skimage.util.view_as_windows(testimg, (19,19), step=step)
psize = patches.shape
# collapse the first 2 dimensions
patches2 = patches.reshape((psize[0]*psize[1], psize[2], psize[3]))
print(patches2.shape )

# histogram equalize patches (improves contrast)
patches3 = empty(patches2.shape)
for i in range(patches2.shape[0]):
    patches3[i,:,:] = skimage.exposure.equalize_hist(patches2[i,:,:])

# extract features
newXf = extract_features(patches3)

现在使用你的分类器进行预测。提取出的特征存放在 `newXf` 中。

In [ ]:
### 在此处填写你的代码
## 提示
## 1. 先对测试数据应用缩放（归一化）

现在我们将在图像上查看结果。使用下面的代码。`prednewY` 是预测结果向量。

In [ ]:
# reshape prediction to an image
imgY = prednewY.reshape(psize[0], psize[1])

# zoom back to image size
imgY2 = ndimage.interpolation.zoom(imgY, step, output=None, order=0)
# pad the top and left with half the window size
imgY2 = vstack((zeros((9, imgY2.shape[1])), imgY2))
imgY2 = hstack((zeros((imgY2.shape[0],9)), imgY2))
# pad right and bottom to same size as image
if (imgY2.shape[0] != testimg.shape[0]):
    imgY2 = vstack((imgY2, zeros((testimg.shape[0]-imgY2.shape[0], imgY2.shape[1]))))
if (imgY2.shape[1] != testimg.shape[1]):
    imgY2 = hstack((imgY2, zeros((imgY2.shape[0],testimg.shape[1]-imgY2.shape[1]))))

# show detections with image
#detimg = dstack(((0.5*imgY2+0.5)*testimg, 0.5*testimg, 0.5*testimg))
nimgY2 = 1-imgY2
tmp = nimgY2*testimg
detimg = dstack((imgY2+tmp, tmp, tmp))

# show it!
plt.figure(figsize=(9,9))
plt.subplot(2,1,1)
plt.imshow(imgY2, interpolation='nearest')
plt.title('detection map')
plt.subplot(2,1,2)
plt.imshow(detimg)
plt.title('image')
plt.axis('image')

_你的人脸检测器效果如何？_
- **在此填写你的答案**

- 你也可以在自己的图像上尝试，不过图像中的人脸大小需要大约为 19x19 像素。
- 我们只用了 1/4 的训练数据。可以尝试用更多数据来训练！